# Initial PEFT Training
- By: **Thakur Tejasva Singh**
* Trained on: Google Colab Nvidia Tesla T4 GPU (16GB VRAM)

In [ ]:
#installing required libraries
%pip install -q -U transformers peft trl bitsandbytes datasets accelerate

In [ ]:
# huggingface login
# huggingface login
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
# Configure 4-bit Quantization and Load the Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-Math-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16  # Force computing inside FP16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16  # CRITICAL: Force base layers to load strictly into FP16! ✅
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Cell 4 - Cleaned up for TRL Integration
from peft import prepare_model_for_kbit_training
import torch

# Prepare model for 4-bit training overhead natively
# This function handles the necessary internal precision adjustments for quantized training
model = prepare_model_for_kbit_training(model)

# Removed the .to(torch.float16) call as it is incompatible with bitsandbytes quantization.
# The model is already loaded with torch_dtype=torch.float16 in the setup cell.

#completely omit get_peft_model here to let SFTTrainer handle it!

In [ ]:
# Load and Format the Dataset
from datasets import load_dataset

# Load the specialized 40K subset
dataset = load_dataset("meta-math/MetaMathQA-40K", split="train")

# Formatting function for Qwen's ChatML-like structure
def format_math_prompt(example):
    text = f"<|im_start|>user\n{example['query']}<|im_end|>\n<|im_start|>assistant\n{example['response']}<|im_end|>"
    return {"text": text}

# Apply the mapping
mapped_dataset = dataset.map(format_math_prompt)

In [ ]:
# Configure T4 VRAM-Optimized Training Arguments (Accelerated for 5-hour limit)
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

training_args = SFTConfig(
    output_dir="./qwen-math-lora-output",
    per_device_train_batch_size=2,      # Increased batch size for speed
    gradient_accumulation_steps=4,      # Adjusted to keep effective batch size similar
    optim="paged_adamw_32bit",
    max_steps=600,                      # Limit steps to finish in ~5 hours
    save_steps=250,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_length=1024,
    report_to="none"
)

In [ ]:
# Final precision fix for T4 GPU compatibility with optimized speed
from trl import SFTTrainer
import torch

# 1. Fresh Trainer initialization with accelerated args
trainer = SFTTrainer(
    model=model,
    train_dataset=mapped_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# 2. Precision fix for T4 (ensuring no BFloat16 in trainable parameters)
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    if "norm" in name:
        param.data = param.data.to(torch.float32)

# 3. Start training (Now set to max_steps=500 for ~5 hour completion)
trainer.train()

In [ ]:
# Cell 7
# Define your target repository name on Hugging Face
new_model_name = "thakur-tejasva/Qwen2.5-Math-7B-PEFT-SOC-Project"

# Save locally to the Colab disk
trainer.model.save_pretrained("final_lora_adapters")

# Push the small adapter weights to Hugging Face
trainer.model.push_to_hub(new_model_name)